# Mass Properties & Bill of Materials
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

Compute the **rigid-body mass properties** of the sized vehicle and produce a
**conceptual Bill of Materials (BOM)** — the last handoff the 6-DoF simulation
(Aetherion) needs before the airframe can fly:

| Output | Consumer |
|--------|----------|
| Total mass \(m\), CG position \(\mathbf{r}_{CG}\) | 6-DoF EOM, `vbat_aero.dml` reference point |
| Inertia tensor \(\mathbf{I}_{CG}\) (body FRD) | 6-DoF EOM, control-authority sizing (NB3 \(\ddot\theta = M/I\)) |
| Principal moments & radii of gyration | sanity checks, handling-qualities estimates |
| `out/bom.csv`, `out/mass_properties.yaml` | procurement, Aetherion / DAVE-ML handoff |

Everything is driven by the **upstream design state** — nothing is hard-coded:

| Input | Source |
|-------|--------|
| Component masses & stations | `out/fuselage.yaml` `layout` (NB4 packing solution) |
| Fuselage meridian \(r(x_s)\) | `fuselage_design.fuselage_radius` (same function as NB4/NB5) |
| Wing span, chord, \(t/c\) | mass closure re-run + `config/wing_structure_params.yaml` |
| Duct / vane geometry | `out/fuselage.yaml`, `out/control_vanes.yaml` (NB3) |

## Axis convention (Aetherion)

Body **FRD** — \(x\) forward (out the nose), \(y\) right, \(z\) down.
Fuselage **stations** \(x_s\) are measured from the nose tip, **positive aft**:
\(x_{body} = -x_s\).  All components sit on the body \(x\)-axis in this
conceptual model, so the CG lies on the axis and the products of inertia
vanish (vane/leg/strut asymmetries are second-order and carried inside the
`misc` fraction).

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import yaml

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design.mass_properties import (
    build_bom, compute_mass_properties, write_mass_properties_yaml,
)
from conceptual_design import reports
from conceptual_design.plots import plot_mass_overview

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` — same pattern as NB2–NB5, so this
notebook stays consistent with the upstream design state — and load the
geometry/packing handoffs written by the previous notebooks.

---

In [ ]:
# -- Re-run the sizing loop + read every upstream handoff ----------------------
inp, result = solve_design_point(CONFIG_PATH)

fus   = load_handoff(OUT_PATH, "fuselage")
vanes = load_handoff(OUT_PATH, "control_vanes")
ail   = load_handoff(OUT_PATH, "aileron")
vib   = load_handoff(OUT_PATH, "vibration")
af    = load_handoff(OUT_PATH, "airfoil")
with open(CONFIG_PATH / "fuselage.yaml") as f:
    fus_cfg = yaml.safe_load(f)
with open(CONFIG_PATH / "modularity.yaml") as f:
    fus_cfg_mod = yaml.safe_load(f)

m_layout = sum(c["mass_kg"] for c in fus["layout"])

print(f"MTOW (mass closure)   : {result.m_total_kg:.4f} kg")
print(f"MTOW (NB4 layout sum) : {m_layout:.4f} kg")
print(f"Fuselage              : D = {fus['D_fus_m']*1e3:.1f} mm, L = {fus['L_fus_m']*1e3:.1f} mm")
print(f"Wing                  : {af['designation']}, b = {result.wing.b_wing*1e3:.0f} mm, "
      f"c = {result.wing.chord_mean*1e3:.1f} mm, "
      f"t = {inp.ws.tc_ratio*result.wing.chord_mean*1e3:.1f} mm")
print(f"Duct                  : {fus['D_duct_inner_m']*1e3:.0f} / "
      f"{fus['D_duct_outer_m']*1e3:.0f} mm, chord {fus['duct_chord_m']*1e3:.0f} mm")
print(f"Axis convention       : {fus['axis_convention']}")

# Section 2 — Inertia Primitives

Each component is idealised as an axisymmetric primitive centred on the body
\(x\)-axis, with inertia known in closed form **about its own CG** (axes
parallel to body FRD).  With everything on the axis, only the diagonal terms
survive and \(I_{yy} = I_{zz}\) for every primitive except the wing plate.

**Solid cylinder** (axis along \(x\), radius \(r\), length \(L\)) — dense bays
(battery, payload, avionics, ESC, motor):

$$
I_{xx} = \tfrac12 m r^2, \qquad
I_{yy} = I_{zz} = \tfrac{1}{12} m \left(3r^2 + L^2\right)
$$

**Thin-walled tube** (duct annulus, mean radius \(r\), length \(L\)):

$$
I_{xx} = m r^2, \qquad
I_{yy} = I_{zz} = m\left(\tfrac{r^2}{2} + \tfrac{L^2}{12}\right)
$$

**Rectangular plate** (wing: span \(b\) along \(y\), chord \(c\) along \(x\),
thickness \(t\) along \(z\)):

$$
I_{xx} = \tfrac{m}{12}\!\left(b^2 + t^2\right),\quad
I_{yy} = \tfrac{m}{12}\!\left(c^2 + t^2\right),\quad
I_{zz} = \tfrac{m}{12}\!\left(b^2 + c^2\right)
$$

**Thin shell of revolution** (fuselage monocoque) — the *same* meridian
\(r(x_s)\) used by NB4 sizing and the NB5 CAD revolve, integrated numerically
with area density \(\sigma\).  For a hoop of width
\(\mathrm{d}s = \sqrt{1 + r'^2}\,\mathrm{d}x\):

$$
\mathrm{d}m = \sigma\, 2\pi r\, \mathrm{d}s, \qquad
\mathrm{d}I_{xx} = r^2\,\mathrm{d}m, \qquad
\mathrm{d}I_{yy}\big|_{ring} = \tfrac{r^2}{2}\,\mathrm{d}m
$$

\(\sigma\) is scaled so the integrated mass matches the NB4
`shell_struct` budget (shell + frames + carry-through, \(k_{struct}\)
included).  Transverse terms are shifted to the shell CG with the parallel
axis theorem, and the whole stack is assembled about the vehicle CG the same
way:

$$
\mathbf{I}_{CG}^{veh} \;=\; \sum_i \Big[ \mathbf{I}_{CG,i}
 \;+\; m_i \big( d_i^2\,\mathbf{E} - \mathbf{d}_i \mathbf{d}_i^{\!\top} \big) \Big],
 \qquad \mathbf{d}_i = \mathbf{r}_i - \mathbf{r}_{CG}
$$

---

In [ ]:
# -- Inertia primitives + assembly live in src/conceptual_design/mass_properties.py.
# Servo-case placement (matches the NB5 CAD): recess below the hub surface
# and case height give the recessed servo-cluster CG radius.
SERVO_RECESS_M = 0.022   # [m] servo case recess below hub surface
SERVO_CASE_H_M = 0.027   # [m] servo case height

mp = compute_mass_properties(
    result, fus, vanes, ail, fus_cfg,
    tc_ratio=inp.ws.tc_ratio,
    servo_recess_m=SERVO_RECESS_M, servo_case_h_m=SERVO_CASE_H_M,
)
print("Component table assembled from the NB6 layout "
      f"({len(mp.components)} primitives).")

# Section 3 — Component Mass Model

Map each entry of the NB4 packing `layout` to a primitive.  Masses and CG
stations come **verbatim** from `out/fuselage.yaml` — this notebook adds only
the *shape* assumption needed for second moments:

| Layout entry | Primitive | Geometry assumption |
|---|---|---|
| `payload`, `avionics`, `battery`, `esc` | solid cylinder | bay length from layout; \(r_{eff} = \sqrt{\eta_{pack}}\,R_{int}\) so the cylinder volume equals the packed component volume |
| `motor_fan` | solid cylinder | \(r = r_{hub}\), \(L = 60\) mm (motor can + fan disc) |
| `duct` | thin tube | mean radius of the annulus, length = duct chord |
| `shell_struct` | shell of revolution | NB4 meridian (shape); CG pinned to the layout station, which includes the \(k_{struct}\) frames allocation |
| `wing` | flat plate | \(b \times c \times t\) from mass closure + \(t/c\) |
| `battery_tray` | solid cylinder | sliding-rail hardware (NB4 `battery_tray_mass_kg`); slides with the battery when the CG trim below is applied |
| **`control_hw`** (vanes + servos + linkages, 4× each) | 3× radial cluster | NB4 places the whole cluster's mass at the aft hinge line (`config/fuselage.yaml` mass model); this notebook only re-splits it into vane / servo / linkage sub-masses at their own radii for the inertia (roll axis) breakdown -- the split must sum back to `layout["control_hw"]["mass_kg"]` exactly |
| **`aileron_hw`** (servo + linkage, 2× each) | radial cluster | NB4 books this at the wing station (`config/aileron.yaml` mass model); this notebook places the pair at their true spanwise moment arm (`out/aileron.yaml` `y_arm_m`) for the roll-inertia contribution -- same pattern as the vane cluster split |
| `misc` (remainder) | solid cylinder | harness, fasteners, bonding, margin; \(r = R_{int}/2\), \(L = L_{fus}/2\) at the layout CG |

The control hardware is placed exactly where NB3 sized it and where the NB5
solid model mounts it: vane mid-span at \(R_{mid}\), vane station `x_vane_m`;
servos at the hinge line \(x_{hinge} = x_{vane,TE} - (1 - x_{hinge}/c)\,c_{vane}\),
CG inside the hub -- **this is now NB4's own model**, not a post-hoc
correction, so the as-packaged CG below should already sit close to the NB4
target. Aileron servo/linkage hardware (NB4, sized in NB4 `aileron_design`)
is booked the same way. The battery-tray trim in Section 4 exists for
whatever (smaller) residual remains -- payload swaps, mass-estimate
refinement, manufacturing tolerance -- bounded by `battery_tray_travel_m`
from `config/fuselage.yaml`.

**Servo selection check** — NB3 requires
\(\tau_{servo} \ge\) `servo_torque_req_gcm` per vane; a 9g-class servo
(≈ 1800 g·cm at 4.8 V) is verified against that below.

---

In [ ]:
# -- Control / aileron hardware split checks (vs the NB6 layout) ---------------
layout = {c["name"]: c for c in fus["layout"]}

servo_torque_avail_gcm = 1800.0          # 9g-class at 4.8 V (e.g. SG90)
tau_req = vanes["servo_torque_req_gcm"]
print(f"Servo torque check : need {tau_req:.0f} g-cm/vane, "
      f"9g-class gives ~{servo_torque_avail_gcm:.0f} g-cm "
      f"-> margin {servo_torque_avail_gcm/tau_req:.1f}x  "
      f"{'OK' if servo_torque_avail_gcm > tau_req else 'FAIL'}")

print(f"Control hardware   : {vanes['n_vanes']} x (vane {mp.m_vane_each*1e3:.1f} g + "
      f"servo {mp.m_servo_each*1e3:.0f} g + linkage {mp.m_link_each*1e3:.0f} g) "
      f"= {mp.m_ctl_total*1e3:.1f} g  (NB4 layout: "
      f"{layout['control_hw']['mass_kg']*1e3:.1f} g)")
print(f"Aileron hardware   : {ail['n_ailerons']} x (servo {mp.m_aileron_servo*1e3:.0f} g + "
      f"linkage {mp.m_aileron_linkage*1e3:.0f} g) = {mp.m_aileron_hw_total*1e3:.1f} g  "
      f"(NB4 layout: {layout['aileron_hw']['mass_kg']*1e3:.1f} g)")
print(f"Joint hardware     : {layout['joint_hw']['mass_kg']*1e3:.1f} g at "
      f"{layout['joint_hw']['x_cg_m']*1e3:.0f} mm (lid hinge + latches)")
print(f"Spar hardware      : {layout['spar_hw']['mass_kg']*1e3:.1f} g "
      f"(d{fus['spar_od_m']*1e3:.0f} x {fus['spar_length_m']*1e3:.0f} mm CFRP tube "
      f"{fus['m_spar_kg']*1e3:.1f} g + 2 root fittings)")
print(f"Isolation hardware : {layout['isolation_hw']['mass_kg']*1e3:.1f} g at "
      f"{layout['isolation_hw']['x_cg_m']*1e3:.0f} mm "
      f"(NB5 -- FC/IMU + payload isolators)")
print()
reports.print_component_table(mp.components)

# Section 4 — Vehicle CG, Battery-Tray Trim, and Inertia Tensor

Assemble the stack about the vehicle CG.  Checks and findings:

1. total mass must equal the mass-closure MTOW (hard assert),
2. off-diagonal terms must be zero — the vane/servo clusters are symmetric
   about the axis (hard assert),
3. the **as-packaged** CG is compared against the NB4 handoff `x_CG_m`.
   NB4 now places the vane/servo/linkage control hardware directly at its
   true aft hinge-line station (`config/fuselage.yaml` mass model) rather
   than lumping it into a mid-body `misc` allocation, so this residual
   should be small — driven only by the minor idealization that NB4 books
   the whole cluster at one station (`x_hinge`) while NB6 splits vanes and
   servos/linkages across their own slightly different stations for the
   inertia breakdown.
4. the battery tray then **solves for the fore-aft slide** that restores
   the NB4 target static margin exactly, checks it against the
   `battery_tray_travel_m` rail limit, and applies it — the mass
   properties and BOM below are the **as-trimmed** (as-delivered) vehicle.

---

In [ ]:
reports.print_cg_trim_report(mp, result, fus)
print("\nMass closure and symmetry checks passed.")

# Section 5 — Visualization

Left: component CGs on the fuselage side profile (marker area ∝ mass), with
the vehicle CG and the wing AC from NB4 — the gap between them *is* the 5 %
static margin.  Right: where each moment of inertia actually comes from
(own-CG term + parallel-axis transport term).

---

In [ ]:
plot_mass_overview(mp, fus, vanes, ail, OUT_PATH / "mass_properties.png")

reports.print_top_iyy_contributors(mp)

# Section 6 — Bill of Materials

Conceptual-level BOM.  Line items follow the NB4 mass budget exactly (so the
`total mass` column sums to MTOW), with the NB3 control-vane hardware and the
landing gear broken out as **indented sub-items whose mass is carried inside
the `misc` / structure fractions** — they get real masses once components are
selected.  Part numbers use `VBT-<CAT>-NNN`
(STR structure, PRP propulsion, PWR power, AVI avionics, PLD payload,
CTL controls, GEA gear).

---

In [ ]:
bom = build_bom(mp, fus, vanes, ail, vib, af, fus_cfg, fus_cfg_mod,
                D_rotor_m=inp.rotor.D_rotor_m,
                servo_torque_avail_gcm=servo_torque_avail_gcm)

print(bom.to_string(index=False, na_rep="(in structure)"))
print("-" * 100)
print(f"BOM mass total : {bom['total_mass_kg'].sum():.4f} kg   (MTOW {mp.m_tot:.4f} kg)")

bom.to_csv(OUT_PATH / "bom.csv", index=False)
print(f"\nWrote {OUT_PATH / 'bom.csv'}")

# Section 7 — Handoff: `out/mass_properties.yaml`

Written for the Aetherion 6-DoF EOM and the DAVE-ML export
(`vbat_aero.dml` reference point).  Regenerate by re-running this notebook.

---

In [ ]:
write_mass_properties_yaml(mp, fus, OUT_PATH / "mass_properties.yaml")

print(f"Wrote {OUT_PATH / 'mass_properties.yaml'}")
print(f"\n  m      = {mp.m_tot:.4f} kg")
print(f"  r_CG   = [{mp.r_cg_body[0]:+.4f}, 0, 0] m  (body FRD, nose origin)")
print(f"  I_CG   = diag({mp.Ixx:.5f}, {mp.Iyy:.5f}, {mp.Izz:.5f}) kg m^2")

# Summary & Limitations

- **Mass closes exactly** against the NB4 budget — the control hardware
  (vanes, 9g-class servos, linkages) is now placed by NB4 itself at its true
  aft hinge-line station (`config/fuselage.yaml` mass model), matching where
  the NB5 solid model mounts it (recessed in the exhaust centerbody hub,
  output shaft on the vane hinge line). This replaces the previous NB6-only
  "carve out of misc" correction, which used to silently move the CG aft of
  the NB4 estimate after the fact.
- **Battery-tray CG trim** now closes the small remaining residual — the
  idealization that NB4 books the whole control-hardware cluster at one
  station (`x_hinge`) while NB6 splits vanes/servos/linkages across their
  own true stations for the inertia breakdown. Section 4 solves for the
  battery-tray slide (`x_battery_trim_m`, bounded by `battery_tray_travel_m`)
  that restores the NB4 target margin, and every downstream quantity (BOM,
  inertia, `out/mass_properties.yaml`) reflects that **as-trimmed** vehicle.
  If a future design change ever needs more trim than the rail travel
  allows, this notebook flags it explicitly instead of silently reporting
  an unstable margin — that is the signal to rebalance the NB4 packing
  itself (a bigger change than sliding the tray) rather than relying on trim.
- **Inertias are shape estimates.**  Axisymmetric primitives are standard at
  the conceptual stage; expect ±15–20 % on \(I_{yy}, I_{zz}\) and more on
  \(I_{xx}\) (small, dominated by duct, wing and the vane/servo clusters).
  The symmetric clusters keep the products of inertia identically zero; the
  real vehicle will have small non-zero terms.
- **Cross-check path:** the NB5 CadQuery solid (now including the four
  servos) can provide an independent uniform-density inertia check per part
  once wall thicknesses are modelled. The battery-tray rails themselves are
  not yet modelled in NB5 — a natural follow-up once this trim scheme is
  validated.
- **Next step:** replace the remaining fraction-based items (`avionics`,
  `esc`, `misc` remainder) with datasheet masses as hardware is selected,
  then re-run NB4 → NB6 to update CG, static margin, and the NB3
  control-authority numbers (\(\ddot\theta = M/I_{yy}\)) in one pass.